# Experiment: Standard Poisson for `basket_size`

**Status: rejected — superseded by the zero/upper-truncated Poisson used in the main notebook**
(`../customer_behavior.ipynb`).

This notebook is kept as a record of the actual modeling path, not as a recommended approach. The
observed mean-variance relationship of `basket_size` initially suggested a standard Poisson
likelihood was reasonable. It fits without error and converges cleanly — but a **standard Poisson
places nonzero probability on basket sizes below 1 and above 9**, values that never occur in a
single coffee-shop transaction. That's not a subtle problem: posterior predictive checks below
show it generating impossible orders. The fix (restricting the support to `[1, 9]`) is a
one-line change — see `zt_poisson` in the main notebook — so there was never a real tradeoff to
keep this version around for production use. It's preserved here purely to document why the
truncation was necessary.

## Setup

This notebook is self-contained and can be run independently of the main notebook.

In [ ]:
import logging
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pymc as pm
import arviz as az
import arviz_plots as azp

logging.getLogger("pymc").setLevel(logging.ERROR)
sns.set_theme(style="white")

RANDOM_SEED = 42

df = pd.read_csv("../../sql/modeling/promo_analysis_dataset.csv")
df_posterior = df.sample(n=100_000, random_state=RANDOM_SEED)
T_i = df_posterior["promotion_used"].to_numpy()
y_obs = df_posterior["basket_size"].to_numpy()

# balanced synthetic treatment vector, used only for prior predictive checks
T_prior = np.array([0] * 500 + [1] * 500)

## Mean-Variance Check

`basket_size` has mean ≈ variance ≈ 4.00 (ratio ≈ 1.00, computed in the main notebook), which is
exactly the condition a Poisson likelihood assumes: $\operatorname{Var}(Y) = E(Y)$. On that basis
alone, a standard Poisson looked like a reasonable first model.

## Model

$$Y_i \sim \operatorname{Poisson}(\mu_i), \qquad \log(\mu_i) = \alpha + \tau T_i$$

$\alpha \sim N(\log 4,\ 0.25)$, $\tau \sim N(0,\ 0.1)$ — the same final priors selected for the
truncated version in the main notebook, so the two models are directly comparable.

In [ ]:
def build_poisson_model(T, y_obs=None, alpha_mu=np.log(4), alpha_sd=0.25, tau_sd=0.1):
    with pm.Model() as model:
        alpha = pm.Normal("alpha", mu=alpha_mu, sigma=alpha_sd)
        tau = pm.Normal("tau", mu=0, sigma=tau_sd)
        mu = pm.math.exp(alpha + tau * T)
        if y_obs is not None:
            pm.Poisson("basket_size", mu=mu, observed=y_obs)
        else:
            pm.Poisson("basket_size", mu=mu, shape=len(T))
    return model

## Posterior

In [ ]:
with build_poisson_model(T_i, y_obs=y_obs) as poisson_model:
    poisson_trace = pm.sample(chains=2, cores=2, random_seed=RANDOM_SEED)

azp.plot_trace_dist(poisson_trace, var_names=["alpha", "tau"])
plt.tight_layout()
plt.show()

display(az.summary(poisson_trace, var_names=["alpha", "tau"], ci_prob=0.94, ci_kind="eti", round_to=3))

In [ ]:
tau_samples = poisson_trace.posterior["tau"].values.flatten()
pct_diff = 100 * (np.exp(tau_samples) - 1)

print(f"Mean percent difference: {pct_diff.mean():.2f}%")
print(f"94% credible interval: {np.quantile(pct_diff, [0.03, 0.97]).round(2)}")
print(f"Probability difference > 0%: {np.mean(pct_diff > 0):.3f}")
print(f"Probability difference >= 5%: {np.mean(pct_diff >= 5):.4f}")

The posterior itself is well-behaved — both parameters converge cleanly ($\hat{R} = 1.00$), and the
promotion-effect conclusion (a small, uncertain difference straddling zero) is consistent with
every other model tried for this outcome. The parameter estimates were never the problem with this
model; the *predictive* behavior is.

## Posterior Predictive Check — Where It Breaks

This is the check that ends the standard Poisson's candidacy.

In [ ]:
trace_ppc = poisson_trace.sel(draw=slice(None, None, 20))
with poisson_model:
    ppc = pm.sample_posterior_predictive(trace_ppc, var_names=["basket_size"], random_seed=RANDOM_SEED)

azp.plot_ppc_pava(ppc, ci_prob=0.90, var_names=["basket_size"])
plt.tight_layout()
plt.show()

azp.plot_ppc_dist(ppc, var_names=["basket_size"], num_samples=20)
plt.tight_layout()
plt.show()

In [ ]:
y_ppc = ppc.posterior_predictive["basket_size"].values.flatten()

print(f"Simulated basket sizes below 1 (impossible order): {np.mean(y_ppc < 1):.4f}")
print(f"Simulated basket sizes above 9 (never observed):    {np.mean(y_ppc > 9):.4f}")
print(f"Simulated basket sizes of exactly 0:                 {np.mean(y_ppc == 0):.4f}")

## Why This Was Rejected

The posterior predictive distribution puts nonzero probability on **zero-item transactions** and
on basket sizes well above the observed maximum of 9 — neither of which can represent a real
coffee-shop order. The calibration plot also shows visible departures from the reference line in
both tails. The observed mean-variance ratio supported Poisson as a *starting point*, but it says
nothing about the plausibility of predictions outside the observed range — that's a separate
constraint the model needs to respect directly.

**Resolution:** restrict the same Poisson likelihood to its observed support with
`pm.Truncated(pm.Poisson.dist(mu=mu), lower=1, upper=9)`. See `zt_poisson` in the main notebook
(`../customer_behavior.ipynb`) for the version actually used to answer the business question.